# DAWN Detectron2 Registration and Validation

This notebook registers the processed DAWN dataset in Pascal VOC format for Detectron2 and verifies that the dataset splits, annotations, and class mappings are correctly loaded before Faster R-CNN training and evaluation.

## Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports

In [16]:
%cd /content
!rm -rf detectron2
!pip uninstall -y detectron2


%cd /content
!git clone https://github.com/facebookresearch/detectron2.git


#  Install Detectron2
%cd /content/detectron2
!python -m pip install --no-build-isolation -e .
%cd /content

/content
/content
Cloning into 'detectron2'...
remote: Enumerating objects: 15962, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 15962 (delta 8), reused 2 (delta 2), pack-reused 15941 (from 2)
Receiving objects: 100% (15962/15962), 6.71 MiB | 16.97 MiB/s, done.
Resolving deltas: 100% (11345/11345), done.
/content/detectron2
Obtaining file:///content/detectron2
  Preparing metadata (setup.py) ... done
  Running setup.py develop for detectron2
/content


In [ ]:
# restart runtime
import os
os.kill(os.getpid(), 9)

In [1]:
# Check import
import sys

sys.path.insert(0, "/content/detectron2")

for m in list(sys.modules.keys()):
    if m.startswith("detectron2"):
        del sys.modules[m]

import detectron2
print("Detectron2 path:", detectron2.__file__)

Detectron2 path: /content/detectron2/detectron2/__init__.py


In [2]:
from detectron2.data.datasets import register_pascal_voc
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.utils.visualizer import Visualizer

print("Detectron2 imports OK")

Detectron2 imports OK


In [3]:
from pathlib import Path
from collections import Counter
import random
import cv2
import matplotlib.pyplot as plt
import torch

### Project Configuration

In [4]:
PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Dissertation")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

output_root_voc = PROJECT_ROOT / "Datasets/processed/dawn_voc"
CLASS_NAMES = ["person", "bicycle", "car", "motorcycle", "bus", "truck"]

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DEVICE       :", DEVICE)
print("VOC ROOT     :", output_root_voc)
print("CLASSES      :", CLASS_NAMES)

PROJECT_ROOT : /content/drive/MyDrive/Colab Notebooks/Dissertation
DEVICE       : cuda
VOC ROOT     : /content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/dawn_voc
CLASSES      : ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']


### Data Structure Check

In [5]:
required_paths = [
    output_root_voc / "Annotations",
    output_root_voc / "JPEGImages",
    output_root_voc / "ImageSets" / "Main",
    output_root_voc / "ImageSets" / "Main" / "train.txt",
    output_root_voc / "ImageSets" / "Main" / "val.txt",
    output_root_voc / "ImageSets" / "Main" / "test_fog.txt",
    output_root_voc / "ImageSets" / "Main" / "test_rain.txt",
    output_root_voc / "ImageSets" / "Main" / "test_snow.txt",
]

for p in required_paths:
    print(f"{p} -> {'OK' if p.exists() else 'MISSING'}")

/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/dawn_voc/Annotations -> OK
/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/dawn_voc/JPEGImages -> OK
/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/dawn_voc/ImageSets/Main -> OK
/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/dawn_voc/ImageSets/Main/train.txt -> OK
/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/dawn_voc/ImageSets/Main/val.txt -> OK
/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/dawn_voc/ImageSets/Main/test_fog.txt -> OK
/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/dawn_voc/ImageSets/Main/test_rain.txt -> OK
/content/drive/MyDrive/Colab Notebooks/Dissertation/Datasets/processed/dawn_voc/ImageSets/Main/test_snow.txt -> OK


### Reset Dataset Registration

In [7]:
def reset_dawn_datasets():
    from detectron2.data import DatasetCatalog, MetadataCatalog

    for name in DatasetCatalog.list():
        if name.startswith("dawn"):
            DatasetCatalog.remove(name)

    for name in MetadataCatalog.list():
        if name.startswith("dawn"):
            MetadataCatalog.remove(name)

    print("All DAWN datasets removed.")

### Register DAWN Splits in Detectron2

In [8]:
reset_dawn_datasets()

split_mapping = {
    "dawn_train": "train",
    "dawn_val": "val",
    "dawn_test_fog": "test_fog",
    "dawn_test_rain": "test_rain",
    "dawn_test_snow": "test_snow",
}

for dataset_name, split_name in split_mapping.items():
    register_pascal_voc(
        name=dataset_name,
        dirname=str(output_root_voc),
        split=split_name,
        year="2007",
        class_names=CLASS_NAMES
    )

print("DAWN VOC datasets registered successfully.")

All DAWN datasets removed.
DAWN VOC datasets registered successfully.


### Check Registered Datasets

In [ ]:
registered = [name for name in DatasetCatalog.list() if name.startswith("dawn")]
print("Registered DAWN datasets:")
for name in registered:
    print("-", name)

Registered DAWN datasets:
- dawn_train
- dawn_val
- dawn_test_fog
- dawn_test_rain
- dawn_test_snow


In [9]:
for dataset_name in split_mapping.keys():
    metadata = MetadataCatalog.get(dataset_name)
    print(dataset_name, metadata.thing_classes)

dawn_train ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
dawn_val ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
dawn_test_fog ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
dawn_test_rain ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
dawn_test_snow ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']


### Inspect Metadata

In [10]:
for dataset_name in split_mapping.keys():
    metadata = MetadataCatalog.get(dataset_name)
    print(f"\n{dataset_name}")
    print("thing_classes:", metadata.thing_classes)


dawn_train
thing_classes: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']

dawn_val
thing_classes: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']

dawn_test_fog
thing_classes: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']

dawn_test_rain
thing_classes: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']

dawn_test_snow
thing_classes: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']


### Check number of images per split

In [11]:
for dataset_name in split_mapping.keys():
    dataset_dicts = DatasetCatalog.get(dataset_name)
    print(f"{dataset_name}: {len(dataset_dicts)} images")

dawn_train: 423 images
dawn_val: 140 images
dawn_test_fog: 60 images
dawn_test_rain: 40 images
dawn_test_snow: 40 images
